<a href="https://colab.research.google.com/github/icosahedron31/Walmart-Sales/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wandb onnx -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 47.2 MB/s eta 0:00:00


In [ ]:
!kaggle --version

Kaggle CLI 2.0.2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
! mkdir ~/.kaggle

In [ ]:
!cp /content/drive/MyDrive/ColabNotebooks/kaggle_API_credentials/kaggle.json ~/.kaggle/kaggle.json

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
try:
    import kaggle
    print("kaggle already installed")
except ImportError:
    !pip install kaggle==1.6.17 -q

kaggle already installed


In [ ]:
!kaggle --version

Kaggle CLI 2.0.2


In [ ]:
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting

100% 2.70M/2.70M [00:00<00:00, 171MB/s]



In [ ]:
! unzip walmart-recruiting-store-sales-forecasting

Archive:  walmart-recruiting-store-sales-forecasting.zip
  inflating: features.csv.zip        
  inflating: sampleSubmission.csv.zip  
  inflating: stores.csv              
  inflating: test.csv.zip            
  inflating: train.csv.zip           


In [ ]:
! unzip features.csv.zip
! unzip test.csv.zip
! unzip train.csv.zip


Archive:  features.csv.zip
  inflating: features.csv            
Archive:  test.csv.zip
  inflating: test.csv                
Archive:  train.csv.zip
  inflating: train.csv               


In [ ]:
! unzip sampleSubmission.csv.zip

Archive:  sampleSubmission.csv.zip
  inflating: sampleSubmission.csv    


**Imports**

In [ ]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
from sklearn.model_selection import train_test_split
import sys
sys.path.append('/content/drive/MyDrive/walmart/Transformers')
from EconomicIndicatorImputer import EconomicIndicatorImputer
from MarkdownImputer import MarkdownImputer
from HolidayEngineer import HolidayProximityTransformer
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cpu


**Reading Data**

In [ ]:
df_sales = pd.read_csv("train.csv")
df_stores = pd.read_csv("stores.csv")
df_features = pd.read_csv("features.csv")

In [ ]:
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  object 
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1)
memory usage: 13.3+ MB


In [ ]:
df_stores.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [ ]:
df_stores.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [ ]:
df_sales.describe()

,Store,Dept,Weekly_Sales
count,421570.000000,421570.000000,421570.000000
mean,22.200546,44.260317,15981.258123
std,12.785297,30.492054,22711.183519
min,1.000000,1.000000,-4988.940000
25%,11.000000,18.000000,2079.650000
50%,22.000000,37.000000,7612.030000
75%,33.000000,74.000000,20205.852500
max,45.000000,99.000000,693099.360000


In [ ]:
df_features["Fuel_Price"].describe()

,Fuel_Price
count,8190.000000
mean,3.405992
std,0.431337
min,2.472000
25%,3.041000
50%,3.513000
75%,3.743000
max,4.468000


In [ ]:
df_features.describe()

,Store,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
count,8190.000000,8190.000000,8190.000000,4032.000000,2921.000000,3613.000000,3464.000000,4050.000000,7605.000000,7605.000000
mean,23.000000,59.356198,3.405992,7032.371786,3384.176594,1760.100180,3292.935886,4132.216422,172.460809,7.826821
std,12.987966,18.678607,0.431337,9262.747448,8793.583016,11276.462208,6792.329861,13086.690278,39.738346,1.877259
min,1.000000,-7.290000,2.472000,-2781.450000,-265.760000,-179.260000,0.220000,-185.170000,126.064000,3.684000
25%,12.000000,45.902500,3.041000,1577.532500,68.880000,6.600000,304.687500,1440.827500,132.364839,6.634000
50%,23.000000,60.710000,3.513000,4743.580000,364.570000,36.260000,1176.425000,2727.135000,182.764003,7.806000
75%,34.000000,73.880000,3.743000,8923.310000,2153.350000,163.150000,3310.007500,4832.555000,213.932412,8.567000
max,45.000000,101.950000,4.468000,103184.980000,104519.540000,149483.310000,67474.850000,771448.100000,228.976456,14.313000


In [ ]:
df_test = pd.read_csv("test.csv")
df_test.head()

,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


In [ ]:
# Check for duplicate keys before merging
print("Duplicate Store rows in stores.csv:", df_stores.duplicated(subset=["Store"]).sum())
print("Duplicate Store/Date rows in features.csv:", df_features.duplicated(subset=["Store", "Date"]).sum())
print("Duplicate Store/Date rows in train_raw:", df_sales.duplicated(subset=["Store", "Date"]).sum())
print(df_sales.duplicated(subset=["Store", "Dept", "Date"]).sum())  # should be 0 or near-0


Duplicate Store rows in stores.csv: 0
Duplicate Store/Date rows in features.csv: 0
Duplicate Store/Date rows in train_raw: 415135
0


In [ ]:
df_features_clean = df_features.drop(columns=['IsHoliday'])
train_full = df_sales.merge(df_features_clean, on=["Store", "Date"], how="left")
train_full = train_full.merge(df_stores, on="Store", how="left")

# Make sure Date is an actual datetime, not a string
train_full["Date"] = pd.to_datetime(train_full["Date"])


In [ ]:
train_full = train_full.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

# e.g. last 20% of the date range becomes validation
cutoff_date = pd.to_datetime("2012-04-01")

X_train = train_full[train_full["Date"] <= cutoff_date].copy()
X_val   = train_full[train_full["Date"] > cutoff_date].copy()

In [ ]:
X_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [ ]:
X_val.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
113,1,1,2012-04-06,57592.12,False,70.43,3.891,10121.97,NaN,77.98,3750.59,4510.72,221.435611,7.143,A,151315
114,1,1,2012-04-13,34684.21,False,69.07,3.891,6186.19,3288.69,17.07,1822.55,1063.78,221.510210,7.143,A,151315
115,1,1,2012-04-20,16976.19,False,66.76,3.877,2230.80,612.02,19.75,275.13,5747.10,221.564074,7.143,A,151315
116,1,1,2012-04-27,16347.60,False,67.23,3.814,3221.25,NaN,35.49,577.14,6222.25,221.617937,7.143,A,151315
117,1,1,2012-05-04,17147.44,False,75.55,3.749,21290.13,NaN,69.89,4977.35,3261.04,221.671800,7.143,A,151315


In [ ]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ("economic_indicator_imputer", EconomicIndicatorImputer()),
    ("markdown_imputer", MarkdownImputer(fill_value=0)),
    #("holiday_proximity", HolidayProximityTransformer(date_col="Date"))
])

X_train_clean = pipeline.fit_transform(X_train)

In [ ]:
X_train_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 332778 entries, 0 to 421539
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         332778 non-null  int64         
 1   Dept          332778 non-null  int64         
 2   Date          332778 non-null  datetime64[ns]
 3   Weekly_Sales  332778 non-null  float64       
 4   IsHoliday     332778 non-null  bool          
 5   Temperature   332778 non-null  float64       
 6   Fuel_Price    332778 non-null  float64       
 7   MarkDown1     332778 non-null  float64       
 8   MarkDown2     332778 non-null  float64       
 9   MarkDown3     332778 non-null  float64       
 10  MarkDown4     332778 non-null  float64       
 11  MarkDown5     332778 non-null  float64       
 12  CPI           332778 non-null  float64       
 13  Unemployment  332778 non-null  float64       
 14  Type          332778 non-null  object        
 15  Size          332778 n

In [ ]:
X_test_t = pipeline.transform(X_val).copy()

In [ ]:
!pip install dagshub
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/

In [ ]:
import dagshub
dagshub.init(repo_owner='icosahedron31', repo_name='Walmart-Sales', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a2e29cde-4a6e-4761-9542-976a903c98c6&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=696117e1a7e79d6d8cdea823d970ed3fa274be9429151265b51d444dd632c83c




Accessing as njvar23

Initialized MLflow to track repo "icosahedron31/Walmart-Sales"

Repository icosahedron31/Walmart-Sales initialized!

In [ ]:

import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name="preprocessing_pipeline"):
    # Fit as usual
    X_train_clean = pipeline.fit_transform(X_train)
    X_val_clean = pipeline.transform(X_val)

    # Log params (whatever config matters)
    mlflow.log_param("econ_columns", list(pipeline.named_steps["economic_indicator_imputer"].columns))
    mlflow.log_param("markdown_fill_value", pipeline.named_steps["markdown_imputer"].fill_value)

    # Log the fitted pipeline itself as an artifact/model
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="preprocessing_pipeline",
        input_example=X_train.head(5),
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE,
    )

2026/07/11 13:32:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 13:32:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can 

🏃 View run preprocessing_pipeline at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/0/runs/58f2684e6ddf4253b1475955e4dd5f93
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/0


In [ ]:
print(mlflow.get_tracking_uri())

https://dagshub.com/icosahedron31/Walmart-Sales.mlflow


In [ ]:
import os
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

model = mlflow.sklearn.load_model(
    "models:/preprocess/1"
)
print("Model loaded successfully")

Model loaded successfully


In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 332778 entries, 0 to 421539
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         332778 non-null  int64         
 1   Dept          332778 non-null  int64         
 2   Date          332778 non-null  datetime64[ns]
 3   Weekly_Sales  332778 non-null  float64       
 4   IsHoliday     332778 non-null  bool          
 5   Temperature   332778 non-null  float64       
 6   Fuel_Price    332778 non-null  float64       
 7   MarkDown1     62298 non-null   float64       
 8   MarkDown2     57920 non-null   float64       
 9   MarkDown3     57266 non-null   float64       
 10  MarkDown4     55777 non-null   float64       
 11  MarkDown5     62640 non-null   float64       
 12  CPI           332778 non-null  float64       
 13  Unemployment  332778 non-null  float64       
 14  Type          332778 non-null  object        
 15  Size          332778 n

In [ ]:
X_val.info()

<class 'pandas.core.frame.DataFrame'>
Index: 88792 entries, 113 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Store         88792 non-null  int64         
 1   Dept          88792 non-null  int64         
 2   Date          88792 non-null  datetime64[ns]
 3   Weekly_Sales  88792 non-null  float64       
 4   IsHoliday     88792 non-null  bool          
 5   Temperature   88792 non-null  float64       
 6   Fuel_Price    88792 non-null  float64       
 7   MarkDown1     88383 non-null  float64       
 8   MarkDown2     53328 non-null  float64       
 9   MarkDown3     79825 non-null  float64       
 10  MarkDown4     79190 non-null  float64       
 11  MarkDown5     88792 non-null  float64       
 12  CPI           88792 non-null  float64       
 13  Unemployment  88792 non-null  float64       
 14  Type          88792 non-null  object        
 15  Size          88792 non-null  int64   

In [ ]:
X_val_c = model.transform(X_val)
X_val_c.info()

<class 'pandas.core.frame.DataFrame'>
Index: 88792 entries, 113 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Store         88792 non-null  int64         
 1   Dept          88792 non-null  int64         
 2   Date          88792 non-null  datetime64[ns]
 3   Weekly_Sales  88792 non-null  float64       
 4   IsHoliday     88792 non-null  bool          
 5   Temperature   88792 non-null  float64       
 6   Fuel_Price    88792 non-null  float64       
 7   MarkDown1     88792 non-null  float64       
 8   MarkDown2     88792 non-null  float64       
 9   MarkDown3     88792 non-null  float64       
 10  MarkDown4     88792 non-null  float64       
 11  MarkDown5     88792 non-null  float64       
 12  CPI           88792 non-null  float64       
 13  Unemployment  88792 non-null  float64       
 14  Type          88792 non-null  object        
 15  Size          88792 non-null  int64   

In [ ]:
# Calendar features
full_data = pd.concat([X_train_clean, X_val_clean]).sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

full_data['Week'] = full_data['Date'].dt.isocalendar().week.astype(int)
full_data['Month'] = full_data['Date'].dt.month
full_data['Year'] = full_data['Date'].dt.year
full_data['Quarter'] = full_data['Date'].dt.quarter

In [ ]:
# Holiday name features
superbowl = pd.to_datetime(['2010-02-12','2011-02-11','2012-02-10','2013-02-08'])
thanksgiving = pd.to_datetime(['2010-11-26','2011-11-25','2012-11-23'])
christmas = pd.to_datetime(['2010-12-31','2011-12-30','2012-12-28'])
laborday = pd.to_datetime(['2010-09-10','2011-09-09','2012-09-07'])

full_data['holiday_name'] = 'none'
full_data.loc[full_data['Date'].isin(superbowl), 'holiday_name'] = 'superbowl'
full_data.loc[full_data['Date'].isin(thanksgiving), 'holiday_name'] = 'thanksgiving'
full_data.loc[full_data['Date'].isin(christmas), 'holiday_name'] = 'christmas'
full_data.loc[full_data['Date'].isin(laborday), 'holiday_name'] = 'laborday'

In [ ]:
# Lag features
grp = full_data.groupby(['Store', 'Dept'])['Weekly_Sales']
full_data['lag_1'] = grp.shift(1)
full_data['lag_2'] = grp.shift(2)
full_data['lag_4'] = grp.shift(4)
full_data['lag_52'] = grp.shift(52)

In [ ]:
# Rolling features
shifted = full_data.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1)
full_data['rolling_mean_4'] = shifted.groupby([full_data['Store'], full_data['Dept']]).transform(lambda x: x.rolling(4).mean())
full_data['rolling_mean_12'] = shifted.groupby([full_data['Store'], full_data['Dept']]).transform(lambda x: x.rolling(12).mean())
full_data['rolling_std_4'] = shifted.groupby([full_data['Store'], full_data['Dept']]).transform(lambda x: x.rolling(4).std())
full_data['rolling_mean_52'] = shifted.groupby([full_data['Store'], full_data['Dept']]).transform(lambda x: x.rolling(52).mean())

In [ ]:
# Drop NaN rows created by lag_52
#full_data = full_data.dropna(subset=['lag_52', 'rolling_mean_52'])
#full_data = full_data.reset_index(drop=True)

In [ ]:
# Run this in your preprocessing notebook
train_check2 = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/train_processed.csv')
print(pd.to_datetime(train_check2['Date']).min())

2010-02-05 00:00:00


In [ ]:
# Split back into train and val
cutoff_date = pd.to_datetime("2012-04-01")
X_train_final = full_data[full_data['Date'] <= cutoff_date].copy()
X_val_final = full_data[full_data['Date'] > cutoff_date].copy()

print(X_train_final.shape)
print(X_val_final.shape)
print(X_train_final.isnull().sum())

(332778, 29)
(88792, 29)
Store                   0
Dept                    0
Date                    0
Weekly_Sales            0
IsHoliday               0
Temperature             0
Fuel_Price              0
MarkDown1               0
MarkDown2               0
MarkDown3               0
MarkDown4               0
MarkDown5               0
CPI                     0
Unemployment            0
Type                    0
Size                    0
Week                    0
Month                   0
Year                    0
Quarter                 0
holiday_name            0
lag_1                3313
lag_2                6591
lag_4               13066
lag_52             158805
rolling_mean_4      13066
rolling_mean_12     38328
rolling_std_4       13066
rolling_mean_52    158805
dtype: int64


In [ ]:
val_dates = pd.to_datetime(X_val_final['Date'])
weekday = val_dates.min().day_name()  # check what day your dates actually fall on
print(weekday)
expected_weeks = pd.date_range(val_dates.min(), val_dates.max(), freq='W-FRI')
print(len(expected_weeks), val_dates.nunique())
missing = expected_weeks.difference(val_dates.unique())
print(missing)

Friday
30 30
DatetimeIndex([], dtype='datetime64[ns]', freq='W-FRI')


In [ ]:
# WMAE metric
import numpy as np

def wmae(y_true, y_pred, is_holiday):
    weights = is_holiday.map({True: 5, False: 1})
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [ ]:
X_train_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 332778 entries, 0 to 421539
Data columns (total 29 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Store            332778 non-null  int64         
 1   Dept             332778 non-null  int64         
 2   Date             332778 non-null  datetime64[ns]
 3   Weekly_Sales     332778 non-null  float64       
 4   IsHoliday        332778 non-null  bool          
 5   Temperature      332778 non-null  float64       
 6   Fuel_Price       332778 non-null  float64       
 7   MarkDown1        332778 non-null  float64       
 8   MarkDown2        332778 non-null  float64       
 9   MarkDown3        332778 non-null  float64       
 10  MarkDown4        332778 non-null  float64       
 11  MarkDown5        332778 non-null  float64       
 12  CPI              332778 non-null  float64       
 13  Unemployment     332778 non-null  float64       
 14  Type             332778 n

In [ ]:
# Save
full = pd.concat([X_train_final, X_val_final])
full.to_csv("/content/drive/MyDrive/ColabNotebooks/full")
X_train_final.to_csv('/content/drive/MyDrive/ColabNotebooks/train_processed.csv', index=False)
X_val_final.to_csv('/content/drive/MyDrive/ColabNotebooks/val_processed.csv', index=False)
print("Saved successfully")

Saved successfully
